# CNN capacity and data scaling

This notebook replaces broad CNN ablations with one controlled question: **how much CNN capacity is useful for a given amount of training data?** The experiment varies training-set size $N$ and a width-scaled CNN family whose actual trainable parameter count is $P$. The fixed `validate` fold is used for early stopping and for the small learning-rate selection inside each $(N,P)$ cell. The test fold is never loaded here.

## Workflow

1. **Memorization sanity check** on one small stratified mini-dataset.
2. **Joint $N \times P$ matrix** using nested stratified subsets and a small inner learning-rate sweep.
3. **Scaling analysis** using tune cross-entropy and the train/tune gap.
4. **Empirical scaling and compute efficiency** using $P_\epsilon(N)$, a descriptive loss surface and the wall-clock frontier.

In [ ]:
from dataclasses import asdict

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

from pytorch_timecourse_classification.artifacts import artifact_root
from pytorch_timecourse_classification.data import load_training_folds
from pytorch_timecourse_classification.experiments import trainable_parameter_count
from pytorch_timecourse_classification.models.cnn import CNNClassifier, scaled_cnn_config
from pytorch_timecourse_classification.scaling import (
    ScalingConfig, compute_efficiency_frontier, fit_scaling_surface,
    near_optimal_parameter_count, run_memorization_check,
    run_scaling_grid, select_best_learning_rate,
)
from pytorch_timecourse_classification.training import TrainingConfig

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
train_fold, tune_fold, preprocessor = load_training_folds()
print("device:", device)
print("train:", tuple(train_fold.features.shape))
print("tune:", tuple(tune_fold.features.shape))
print("classes:", preprocessor.class_names)

## Fixed CNN family

Only convolutional channel width and the hidden width of the classification head scale. Depth, kernels `(9, 7, 5)`, dilation, pooling and the target task remain fixed, so $P$ is a cleaner capacity axis than changing depth.

In [ ]:
BASE_MODEL_CONFIG = {
    "kernel_sizes": (9, 7, 5),
    "dilations": (1, 1, 1),
    "pool_size": 2,
    "dropout": 0.2,
    "residual": False,
}
WIDTH_MULTIPLIERS = (0.25, 0.5, 1.0, 2.0, 4.0)

rows = []
for width in WIDTH_MULTIPLIERS:
    cfg = scaled_cnn_config(width)
    model = CNNClassifier(
        input_length=train_fold.features.shape[-1],
        num_classes=len(preprocessor.class_names),
        **BASE_MODEL_CONFIG, **cfg,
    )
    rows.append({
        "width_multiplier": width,
        "channels": cfg["channels"],
        "hidden_dim": cfg["hidden_dim"],
        "parameter_count": trainable_parameter_count(model),
    })
display(pd.DataFrame(rows))

## 1. Memorization sanity check

This is diagnostic, not model selection. A small **stratified** sample is trained without dropout, weight decay, scheduler or gradient clipping. Failure to reach near-perfect training accuracy should be resolved before interpreting scaling curves.

In [ ]:
memorization = run_memorization_check(
    model_type=CNNClassifier,
    training_fold=train_fold,
    preprocessor=preprocessor,
    base_model_config=BASE_MODEL_CONFIG,
    width_multiplier=1.0,
    sample_count=32,
    epochs=200,
    learning_rate=3e-3,
    device=device,
)
display(pd.Series(asdict(memorization)))
if not memorization.passed:
    raise RuntimeError("Memorization check failed; resolve trainability first.")

## 2. Joint $N \times P$ matrix

Training subsets are nested and stratified. For each $(N,P)$ cell, three learning rates are tried from the same model seed and the minimum validation cross-entropy selects the learning rate. The scheduler is disabled so the inner sweep remains interpretable rather than becoming another broad tuning exercise.

In [ ]:
SCALING_CONFIG = ScalingConfig(
    data_fractions=(0.0625, 0.125, 0.25, 0.5, 1.0),
    width_multipliers=WIDTH_MULTIPLIERS,
    learning_rates=(3e-4, 1e-3, 3e-3),
    subset_seed=42,
    model_seed=42,
)
BASE_TRAINING_CONFIG = TrainingConfig(
    epochs=100, batch_size=256, learning_rate=1e-3,
    weight_decay=1e-4, scheduler_strategy="none",
    patience=15, report_every=20, random_seed=42,
    class_balance_strategy="none", gradient_clip_norm=1.0,
)
print(SCALING_CONFIG)
print(BASE_TRAINING_CONFIG)

In [ ]:
scaling_dir = artifact_root() / "cnn_scaling"
scaling_dir.mkdir(parents=True, exist_ok=True)
runs_path = scaling_dir / "np_lr_grid.csv"
RECOMPUTE = False

if RECOMPUTE or not runs_path.exists():
    runs = run_scaling_grid(
        model_type=CNNClassifier, training_fold=train_fold,
        tuning_fold=tune_fold, preprocessor=preprocessor,
        base_model_config=BASE_MODEL_CONFIG,
        base_training_config=BASE_TRAINING_CONFIG,
        scaling_config=SCALING_CONFIG, device=device, verbose=True,
    )
    runs.to_csv(runs_path, index=False)
else:
    runs = pd.read_csv(runs_path)
    print("loaded cached grid:", runs_path)
print(f"runs={len(runs)}, failed={int(runs['failed'].sum())}")

In [ ]:
selected_runs = select_best_learning_rate(runs)
selected_runs.to_csv(scaling_dir / "np_selected_lr.csv", index=False)
display(selected_runs[[
    "n_samples", "parameter_count", "width_multiplier",
    "learning_rate", "training_loss", "tune_loss",
    "training_macro_f1", "tune_macro_f1",
    "best_epoch", "duration_seconds",
]])

## 3. Scaling analysis

Cross-entropy is the primary scaling quantity because thresholded classification metrics can saturate while probabilistic fit still changes. The train-to-tune loss gap distinguishes useful capacity from additional memorization capacity.

In [ ]:
figure, axis = plt.subplots(figsize=(7.5, 4.5))
for n_samples, group in selected_runs.groupby("n_samples", sort=True):
    group = group.sort_values("parameter_count")
    axis.plot(group["parameter_count"], group["tune_loss"], marker="o", label=f"N={n_samples:,}")
axis.set_xscale("log")
axis.set(title="Model scaling at fixed data size", xlabel="trainable parameters P", ylabel="tune cross-entropy")
axis.grid(alpha=0.2)
axis.legend()
figure.tight_layout()

In [ ]:
loss_matrix = selected_runs.pivot(index="n_samples", columns="parameter_count", values="tune_loss").sort_index().sort_index(axis=1)
gap = selected_runs.assign(loss_gap=selected_runs["tune_loss"] - selected_runs["training_loss"])
gap_matrix = gap.pivot(index="n_samples", columns="parameter_count", values="loss_gap").sort_index().sort_index(axis=1)

for title, matrix in (("Tune cross-entropy", loss_matrix), ("Tune - train loss", gap_matrix)):
    figure, axis = plt.subplots(figsize=(8, 4.5))
    image = axis.imshow(matrix.to_numpy(), aspect="auto", origin="lower")
    axis.set(title=title, xlabel="P", ylabel="N", xticks=np.arange(len(matrix.columns)), yticks=np.arange(len(matrix.index)), xticklabels=[f"{v:,}" for v in matrix.columns], yticklabels=[f"{v:,}" for v in matrix.index])
    axis.tick_params(axis="x", labelrotation=40)
    figure.colorbar(image, ax=axis)
    figure.tight_layout()

display(selected_runs.pivot(index="n_samples", columns="parameter_count", values="learning_rate"))

## 4. Empirical scaling and compute efficiency

For each $N$, $P_\epsilon(N)$ is the smallest model within an absolute tune-loss tolerance of the best observed model. The fitted surface $L(N,P)=L_\infty + A N^{-\alpha} + B P^{-\beta}$ is descriptive for this dataset, not a claim of a universal scaling law.

In [ ]:
near_optimal = near_optimal_parameter_count(selected_runs, tolerance=0.02)
display(near_optimal)
figure, axis = plt.subplots(figsize=(6.5, 4))
axis.plot(near_optimal["n_samples"], near_optimal["parameter_count"], marker="o")
axis.set_xscale("log"); axis.set_yscale("log")
axis.set(title=r"Smallest near-optimal model $P_\epsilon(N)$", xlabel="training samples N", ylabel="trainable parameters P")
axis.grid(alpha=0.2)
figure.tight_layout()

In [ ]:
scaling_fit = fit_scaling_surface(selected_runs)
display(pd.Series(asdict(scaling_fit), name="descriptive scaling fit"))

In [ ]:
frontier = compute_efficiency_frontier(selected_runs)
figure, axis = plt.subplots(figsize=(7, 4.5))
axis.scatter(selected_runs["duration_seconds"], selected_runs["tune_loss"], alpha=0.7, label="N x P cells")
axis.plot(frontier["duration_seconds"], frontier["tune_loss"], marker="o", label="wall-clock frontier")
axis.set(title="Compute-efficient frontier", xlabel="training wall time [s]", ylabel="tune cross-entropy")
axis.grid(alpha=0.2); axis.legend(); figure.tight_layout()
display(frontier[["n_samples", "parameter_count", "learning_rate", "tune_loss", "duration_seconds", "peak_memory_bytes"]])

### Interpretation

- **Capacity-limited:** increasing $P$ lowers both train and tune loss at fixed $N$.
- **Data-limited / over-capacity:** increasing $P$ lowers train loss but not tune loss, while increasing $N$ helps.
- **Optimization-limited:** larger models retain unexpectedly high train loss or select a different learning-rate regime.
- **Compute-efficient choice:** prefer the smallest/cheapest model close to the best tune loss rather than the largest model by default.

The test fold remains untouched. A CNN selected here should only be compared against the extended-CNN and attention families in the later model-family comparison workflow.